In [1]:
2+2

4

In [2]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
import seaborn as sns
import matplotlib.pyplot as plt

import ast
import re


In [3]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [22]:
result = pd.read_parquet("results/logits_output_meta.parquet")
print(result.shape)
result.head(3)

(150, 12)


,video_id,cam,session,template,label,top1_pred,top1_class,top1_conf,top5_preds,top5_classes,correct_top1,correct_top5
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.71,"[49, 106, 1, 145, 5]","[Plugging [something] into [something], Putting [something] behind [something], Attaching [something] to [something], Tearing [something] just a little bit, Closing [something]]",True,True
1,260922-134214,cam1,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.77,"[49, 106, 1, 145, 5]","[Plugging [something] into [something], Putting [something] behind [something], Attaching [something] to [something], Tearing [something] just a little bit, Closing [something]]",True,True
2,260922-134214,cam2,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.89,"[49, 1, 106, 96, 50]","[Plugging [something] into [something], Attaching [something] to [something], Putting [something] behind [something], Pushing [something] onto [something], Plugging [something] into [something] but pulling it right out as you remove your hand]",True,True


In [23]:
result.session.value_counts()

session
plug_1      15
plug_2      15
magnet_1    15
magnet_2    15
books_1     15
books_2     15
lady_1      15
lady_2      15
paper_1     15
paper_2     15
Name: count, dtype: int64

In [12]:
d = np.load("results/logits_output.npz", allow_pickle=True)
print(d["logits"].shape)   # (151, 174)
d

(150, 174)


NpzFile 'results/logits_output.npz' with keys: logits, probs, labels, video_ids, cams...

In [16]:
for k in d.keys():
    print(k, d[k].shape)

logits (150, 174)
probs (150, 174)
labels (150,)
video_ids (150,)
cams (150,)
templates (150,)
class_names (174,)


# main result

In [17]:
from sklearn.metrics import balanced_accuracy_score
    
n = len(result)
top1_acc  = result["correct_top1"].mean()
top5_acc  = result["correct_top5"].mean()
ba_top1   = balanced_accuracy_score(result["label"], result["top1_pred"])

# Top-5 balanced: credit correct if label is anywhere in top-5
top5_pred_adj = [
    row["label"] if row["correct_top5"] else row["top5_preds"][0]
    for _, row in result.iterrows()
]
ba_top5 = balanced_accuracy_score(result["label"], top5_pred_adj)

my_metrics = {
    "total_evaluated":        n,
    "top1_accuracy":          round(top1_acc, 6),
    "top1_balanced_accuracy": round(ba_top1,  6),
    "top5_accuracy":          round(top5_acc, 6),
    "top5_balanced_accuracy": round(ba_top5,  6),
}
my_metrics

/user/h.siddiqui/u27673/.conda/envs/vbbf/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/user/h.siddiqui/u27673/.conda/envs/vbbf/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


{'total_evaluated': 150,
 'top1_accuracy': np.float64(0.246667),
 'top1_balanced_accuracy': np.float64(0.246667),
 'top5_accuracy': np.float64(0.466667),
 'top5_balanced_accuracy': np.float64(0.466667)}

In [ ]:
result.groupby("cam")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
)

,top1_acc,top5_acc
cam,,
cam0,0.26,0.48
cam1,0.24,0.46
cam2,0.24,0.46


In [27]:
result.groupby("template")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
).sort_values("template")

,top1_acc,top5_acc
template,,
Attaching [something] to [something],0.40,0.87
Moving [something] up,0.00,0.00
Piling [something] up,0.00,1.00
Plugging [something] into [something],1.00,1.00
Pushing [something] with [something],1.00,1.00
Putting [something] next to [something],0.00,0.00
Putting [something] on a surface,0.00,0.00
Putting [something] onto [something],0.07,0.80
Showing a photo of [something] to the camera,0.00,0.00


In [26]:
result.groupby("session")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
).sort_values("session")

,top1_acc,top5_acc
session,,
books_1,0.00,1.00
books_2,0.27,0.40
lady_1,0.00,0.00
lady_2,0.20,0.40
magnet_1,0.40,0.87
magnet_2,0.20,0.27
paper_1,0.00,0.00
paper_2,0.20,0.40
plug_1,1.00,1.00


# Granular analysis